In [ ]:
import os

from Libraries.log_helper import LOG
LOG.set_logging_level('DEBUG')

import Libraries.insta360 as insta360


# The workflow around Insta360 X3 video

Lots of stuff to accomplish

- Copy the BBlogs and Videos to their directories.
- Read the metadata, including the IMU data.
- Compute the poses for the videos.
- Recompress and/or combine the videos.
    - Save this in the web directory.
- Extract the boat brain logs which align with the videos.
    - Sync them accurately
- Copy the poses and logs to the web directory.
- Create the videos.json file in the web dir.
- Sync the web dir to cloud.



The new framework I am trying is really a simplification.  All the files have the same basename
(which comes from the X3).  

```
-rw-r--r--  1 viola  staff         2368 May 17 09:29 VID_20240417_183646_00_003.exif
-rw-r--r--  1 viola  staff         4729 May 17 10:14 VID_20240417_183646_00_003.ffprobe
-rw-r--r--  1 viola  staff     34930240 May 17 09:29 VID_20240417_183646_00_003.imu
-rwx------@ 1 viola  staff  10942392580 Apr 17 19:05 VID_20240417_183646_00_003.insv
-rw-r--r--  1 viola  staff      3124816 May 20 09:43 VID_20240417_183646_00_003.log
-rw-r--r--  1 viola  staff      1392880 May 20 08:57 VID_20240417_183646_00_003.poses
```

The first is the captured video (which is really an MP4 with extra metadata).

- The metadata output of exiftool on this file.
- The metadata output of ffprobe.
- The raw extracted IMU data.
    - Exiftool, by default, tries to print this out in a friendly format.  Its crazy slow (minutes).
      I hacked the exiftool to just dump the bytes directly (second?).
    - This data can be read directly in python (numpy), once again in seconds!
- The X3 video file.
- The synced sensor logs collected by Boat Brain
- The camera poses estimated from the IMU data.


## Insta360 accel gyro data

Videos are captured in 29 min blocks.  

      Duration: 00:29:00.94, start: 0.000000, bitrate: 40357 kb/s

There appears to be a 15 second gap between files?!?!

    VID_20240422_180317_10_001.insv
    VID_20240422_183232_00_002.insv

(This is supposedly well known.)

There is also a "create time" stored in the metadata (exiftool or ffprobe).  Interestingly, **it is
not the same at the filename**?!?!

The best documentation I have on the accel/gyro data is from exiftool.  The data is stored in a binary blob with

- timecode:  unsigned 64bit in microseconds
- accelerometer:  3 unsigned shorts
    - Recenter and then downscale by 1000
- gyro: 3 ushort, same as above

The data is available at 1000Hz.  Extraction with exiftool took forever (using Perl to print
floats).  So I hacked exiftool to write the binary blob directly to a file (very fast).  Conversion
using numpy is insanely fast!

## Timing

**Timecode sucks.**

I am struggling with timecode.  The imu timecode seems to continue rolling as you continue to capture
video.  And it is not clear how it relates to framenumber.

**Filenames and creation times are not the same. **

The current plan (implemented below), do a rough extract of the BB logs (which is not synced).  Then
use the compass (from BB) and the gyro (from X3) to compute an alignment.  

In general it appears to be about 15 seconds.

### GPS data

Its possible to sync your phone to the X3, and it will pull across GPS data.  This is potentially
nice because it includes GPS time as well.

There is also a GPS enabled remote for $75 bucks.





In [ ]:
import importlib

In [ ]:

importlib.reload(insta360)
# Managment and Reompression of Video Files.


In [ ]:
insta360.copy_video_from_card()

# Read and cache the metadata for each insv video file.
insta360.extract_video_metadata()
_ = insta360.create_videos_json()

insta360.compress_all_videos()


In [ ]:
importlib.reload(insta360)
insta360.extract_video_metadata()

In [ ]:
import telemetry_parser


In [ ]:
telemetry_parser.Parser()